In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from scipy.stats import genpareto, truncpareto

In [2]:
with open("../../output/arrival_distributions/truncpareto_all_risk.yaml") as f:
  tpareto_config = yaml.safe_load(f)
  
with open("../../output/arrival_distributions/genpareto_all_risk_trunc_spflu.yaml") as f:
  genpareto_spflu_config = yaml.safe_load(f)
  
with open("../../output/econ_loss_models/poisson_model.yaml") as f:
  poisson_config = yaml.safe_load(f)
  
with open("../../output/econ_loss_models/loglog_model.yaml") as f:
  loglog_config = yaml.safe_load(f)

In [3]:
r = 0.04
y = 0.016
Y0 = 12316.1
P0 = 7.91e9
VSL = 1.3e6

In [4]:
# Create truncated Pareto distribution
tpareto_dist = truncpareto(
    b=tpareto_config['params']['b'],
    loc=tpareto_config['params']['loc'], 
    scale=tpareto_config['params']['scale'],
    c=tpareto_config['params']['c']
)

# Create generalized Pareto distribution truncated at Spanish Flu
genpareto_trunc = genpareto_spflu_config['max_severity']
genpareto_dist = genpareto(
    c=genpareto_spflu_config['params']['k'],
    loc=genpareto_spflu_config['params']['theta'],
    scale=genpareto_spflu_config['params']['sigma']
)

In [5]:

def get_severity_tpareto(p):
    """
    Get severity level from truncated Pareto distribution at probability p,
    accounting for arrival rate
    
    Args:
        p (array-like): Probability level(s) between 0 and 1
        
    Returns:
        array-like: Severity level(s) at given probability(ies), 0 if below arrival threshold
    """
    p = np.asarray(p)
    # Account for arrival rate threshold
    result = np.zeros_like(p)
    cum_prob_at_threshold = 1 - tpareto_config['arrival_rate']
    mask = p > cum_prob_at_threshold
    
    # Rescale probability to conditional probability given arrival for values above threshold
    p_cond = np.where(mask,
                      (p - cum_prob_at_threshold) / tpareto_config['arrival_rate'],
                      0)
    result[~mask] = tpareto_config['params']['loc'] + tpareto_config['params']['scale']
    result[mask] = tpareto_dist.ppf(p_cond[mask])
    return result


def get_severity_genpareto(p):
    """
    Get severity level from truncated generalized Pareto distribution at probability p,
    accounting for arrival rate
    
    Args:
        p (array-like): Probability level(s) between 0 and 1
        
    Returns:
        array-like: Severity level(s) at given probability(ies), truncated at Spanish Flu level,
                   0 if below arrival threshold
    """
    p = np.asarray(p)
    # Account for arrival rate threshold
    result = np.zeros_like(p)
    cum_prob_at_threshold = 1 - genpareto_spflu_config['arrival_rate']
    mask = p > cum_prob_at_threshold
    
    # Rescale probability to conditional probability given arrival for values above threshold
    p_cond = np.where(mask,
                      (p - cum_prob_at_threshold) / genpareto_spflu_config['arrival_rate'],
                      0)
    
    result[~mask] = genpareto_spflu_config['min_severity']
    result[mask] = genpareto_dist.ppf(p_cond[mask])
    return np.minimum(result, genpareto_trunc)


def get_gdp_loss_loglog(severity):
    """
    Calculate share of GDP loss using log-log regression model
    
    Args:
        severity (array-like): Pandemic severity levels (deaths per 10,000)
        
    Returns:
        array-like: Share of GDP loss for each severity level
    """
    severity = np.asarray(severity)
    log_severity = np.log(severity)
    log_share_loss = loglog_config['params']['coefs'][0] * log_severity + loglog_config['params']['intercept']
    return np.exp(log_share_loss)


def get_gdp_loss_poisson(severity):
    """
    Calculate share of GDP loss using Poisson regression model
    
    Args:
        severity (array-like): Pandemic severity levels (deaths per 10,000)
        
    Returns:
        array-like: Share of GDP loss for each severity level
    """
    severity = np.asarray(severity)
    log_severity = np.log(severity)
    log_share_loss = log_severity * poisson_config['params']['coefs'] + poisson_config['params']['intercept']
    return np.exp(log_share_loss)


def get_learning_losses(econ_loss):
    """
    Calculate learning losses from economic losses
    
    Args:
        econ_loss (array-like): Economic losses
        
    Returns:
        array-like: Learning losses
    """
    return (10 / 13.8) * econ_loss


In [6]:
def get_intensity(p, mu_prime=1e-3, alpha=0.62, sigma=0.0113, xi=1.41, mu_double_prime=17.8):
    p = np.asarray(p)
    # Account for arrival rate threshold
    result = np.zeros_like(p)
    cum_prob_at_threshold = 1 - 0.35  # arrival rate of 0.35
    mask = p > cum_prob_at_threshold
    
    # Rescale probability to conditional probability given arrival for values above threshold
    p_cond = np.where(mask,
                      (p - cum_prob_at_threshold) / 0.35,
                      0)
   
    # Calculate intensities for values above threshold
    intensity = np.where(
        p_cond <= alpha,
        mu_prime,
        genpareto.ppf((1 - p_cond) / (1 - alpha), c=xi, scale=sigma, loc=mu_prime)
    )
    intensity = np.where(intensity >= mu_double_prime, mu_double_prime, intensity)
    
    # Set values below threshold to minimum intensity
    result[~mask] = 0
    result[mask] = intensity[mask]
    return result


def intensity_econ_model(intensity):
		return np.exp(0.74 + 0.46 * np.log(intensity)) / 100

In [7]:
# Set random seed for reproducibility
np.random.seed(42)

# Generate uniform random variables
n_samples = 1_000_000
u = np.random.uniform(0, 1, n_samples)

# Initialize arrays to store results for each model combination
results = {
    'genpareto_loglog': {'mortality': [], 'economic': [], 'learning': [], 'total': []},
    'genpareto_poisson': {'mortality': [], 'economic': [], 'learning': [], 'total': []},
    'truncpareto_loglog': {'mortality': [], 'economic': [], 'learning': [], 'total': []},
    'truncpareto_poisson': {'mortality': [], 'economic': [], 'learning': [], 'total': []},
    'intensity_model': {'mortality': [], 'economic': [], 'learning': [], 'total': []}
}

# Calculate losses for each sample and model combination
for model_combo in results.keys():
    if model_combo == 'intensity_model':
        # Handle intensity model separately
        intensities = get_intensity(u)
        # Convert from deaths per 1,000 to deaths per 10,000
        severities = intensities * 10
        gdp_losses = intensity_econ_model(intensities)
    else:
        sev_model, econ_model = model_combo.split('_')
        
        # Get severities
        if sev_model == 'genpareto':
            severities = get_severity_genpareto(u)
        else:
            severities = get_severity_tpareto(u)
        
        # Get economic losses
        if econ_model == 'loglog':
            gdp_losses = get_gdp_loss_loglog(severities)
        else:
            gdp_losses = get_gdp_loss_poisson(severities)
    
    # Calculate mortality losses (assuming value of death = $7m and initial population = 330m)
    mortality_losses = severities * (P0/1e4) * VSL
    
    # Calculate economic losses in dollars (assuming US GDP ≈ $21T)
    econ_losses = gdp_losses * Y0 * P0
    
    # Calculate learning losses
    learning_losses = get_learning_losses(econ_losses)
    
    # Store results
    results[model_combo]['mortality'] = np.mean(mortality_losses) * r / (r - y)
    results[model_combo]['economic'] = np.mean(econ_losses) * r / (r - y)
    results[model_combo]['learning'] = np.mean(learning_losses) * r / (r - y)
    results[model_combo]['total'] = np.mean(mortality_losses + econ_losses + learning_losses) * r / (r - y)

# Create formatted table using pandas
df_results = pd.DataFrame.from_dict(results, orient='index')

# Format numbers to billions with 2 decimal places
for col in df_results.columns:
    df_results[col] = df_results[col] / 1e9
    df_results[col] = df_results[col].astype(int)

# Rename columns and index
df_results.columns = ['Mortality Losses ($B)', 'Economic Losses ($B)', 'Learning Losses ($B)', 'Total Losses ($B)']
df_results.index = [
    'Pareto sudden trunc + LogLog', 
    'Pareto sudden trunc + Poisson',
    'TruncPareto + LogLog',
    'TruncPareto + Poisson',
    '~Original IMF model'
]

# Display results
print("\nExpected Annual Pandemic Losses (Billions USD)")
print("=" * 80)
display(df_results)



Expected Annual Pandemic Losses (Billions USD)


C:\Users\squaade\AppData\Local\Temp\ipykernel_26960\2294631448.py:28: RuntimeWarning: divide by zero encountered in log
  return np.exp(0.74 + 0.46 * np.log(intensity)) / 100


,Mortality Losses ($B),Economic Losses ($B),Learning Losses ($B),Total Losses ($B)
Pareto sudden trunc + LogLog,9223,1254,909,11387
Pareto sudden trunc + Poisson,9223,2303,1669,13196
TruncPareto + LogLog,146712,4052,2936,153701
TruncPareto + Poisson,146712,2757,1998,151468
~Original IMF model,515,143,103,762
